# Explainable AI (XAI): LIME and Generative AI
Explainable AI (XAI) helps us understand how complex models make decisions. In this lab, we'll explore XAI using LIME, and consider how explainability applies to generative AI. We'll work with a topical dataset of tweets labelled by emotion, providing a relatable context for learners.

## Dataset Overview
We are using the `dair-ai/emotion` dataset from Hugging Face, which contains tweets labelled with one of six emotions:
- Joy
- Sadness
- Anger
- Fear
- Love
- Surprise

This kind of data is useful in marketing, brand monitoring, and content moderation, making it highly relevant to real-world business applications.

In [6]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 50.1 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3


In [2]:
!pip install --only-binary :all: pyarrow datasets

  Using cached datasets-4.6.1-py3-none-any.whl.metadata (19 kB)
INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 65.4 MB/s  0:00:00
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 37.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 93.0 MB/s  0:00:00
Using cached httpx-0.28.1-py3-no

In [1]:
!pip install -qqq datasets lime shap scikit-learn matplotlib seaborn

ERROR: Failed to build 'pyarrow' when installing build dependencies for pyarrow


In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import lime.lime_text
import shap
import matplotlib.pyplot as plt

In [ ]:
dataset = load_dataset('dair-ai/emotion', split='train')
df = pd.DataFrame(dataset)
df = df[['text', 'label']]
label_map = dataset.features['label'].int2str
df['label'] = df['label'].apply(label_map)
df.head()

## Modelling Emotions in Tweets
We'll vectorise the tweet text and train a Random Forest classifier to predict the emotion.

In [ ]:
X = df['text']
y = df['label']

vectorizer = TfidfVectorizer(max_features=1000)
X_vec = vectorizer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

## LIME for Text Classification
LIME helps us interpret individual predictions on a tweet by approximating the model locally with a simpler one.

In [ ]:
explainer = lime.lime_text.LimeTextExplainer(class_names=model.classes_)
i = 42 #Select different index from dataframe of tweet text
sample_text = X.iloc[i]
exp = explainer.explain_instance(sample_text,
                                 classifier_fn=lambda x: model.predict_proba(vectorizer.transform(x)),
                                 num_features=6,
                                 labels=[model.classes_.tolist().index(y.iloc[i])],
                                 num_samples=500)
print(f"Text: {sample_text}")
exp.show_in_notebook(text=True)

## LIME for Text Classification
Input your own tweet to see how it gets classified and the explore the LIME explinations behind this.

In [5]:
from lime.lime_text import LimeTextExplainer

explainer = LimeTextExplainer(class_names=model.classes_)
user_input = input("Type a tweet to explain (or press Enter to use a default one): ").strip()

if user_input:
    sample_text = user_input
    use_top_label = True
else:
    i = 42
    sample_text = X.iloc[i]
    true_label = model.classes_.tolist().index(y.iloc[i])
    use_top_label = False

exp = explainer.explain_instance(
    sample_text,
    classifier_fn=lambda x: model.predict_proba(vectorizer.transform(x)),
    num_features=6,
    labels=None if use_top_label else [true_label],
    top_labels=1 if use_top_label else None,
    num_samples=500
)

print(f"\nText: {sample_text}")
exp.show_in_notebook(text=True)


ModuleNotFoundError: No module named 'lime'

## Explainability in Generative AI
Generative models such as GPT produce text by predicting the next word. Understanding these models can involve:
- Attention visualisation
- Feature attribution for input tokens
- Tracing prompt impact

Though LIME is not typically used for full language models, ideas of attribution and transparency remain vital.

## Activity: Reflection
Think of a time you saw content on social media that triggered a strong emotional response. If an AI flagged or filtered it, how would you want the decision to be explained?



## Summary
- We applied LIME to interpret an emotion classifier trained on tweets.
- LIME provided individual-level explanations.
- Understanding how models behave with social data is vital in real-world applications like moderation, advertising, and engagement analytics.